In [ ]:
# Write a program to smooth the input image using:  
# 1. Averaging filter (un-weighted)  
# 2.   Weighted filter given by h(x,y)= max(|x|,|y|) 
# 3. Gaussian filter  
# Note: Input the filter size and other parameters such as sigma in case of Gaussian filter, method of padding (replication, zero padding, etc.) from the user. 

import cv2
import numpy as np
import matplotlib.pyplot as plt

def apply_padding(image, pad, padding_type):

    if padding_type == 1:
        padded = np.pad(
            image,
            ((pad, pad), (pad, pad)),
            mode='constant',
            constant_values=0
        )

    elif padding_type == 2:
        padded = np.pad(
            image,
            ((pad, pad), (pad, pad)),
            mode='edge'
        )

    elif padding_type == 3:
        padded = np.pad(
            image,
            ((pad, pad), (pad, pad)),
            mode='reflect'
        )

    else:
        print("Invalid padding choice!")
        print("Using Zero Padding.")
        padded = np.pad(
            image,
            ((pad, pad), (pad, pad)),
            mode='constant',
            constant_values=0
        )

    return padded

def averaging_filter(image, filter_size, padding_type):

    pad = filter_size // 2

    padded = apply_padding(
        image,
        pad,
        padding_type
    )

    kernel = np.ones(
        (filter_size, filter_size),
        dtype=float
    )

    kernel = kernel / (filter_size * filter_size)

    output = np.zeros_like(
        image,
        dtype=float
    )

    rows, cols = image.shape

    for i in range(rows):
        for j in range(cols):

            region = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.sum(
                region * kernel
            )

    return np.uint8(
        np.clip(output, 0, 255)
    )

def weighted_filter(image, filter_size, padding_type):

    pad = filter_size // 2

    padded = apply_padding(
        image,
        pad,
        padding_type
    )

    kernel = np.zeros(
        (filter_size, filter_size),
        dtype=float
    )
    for x in range(-pad, pad + 1):

        for y in range(-pad, pad + 1):

            kernel[
                x + pad,
                y + pad
            ] = max(
                abs(x),
                abs(y)
            )
    kernel[pad, pad] = 1

    kernel = kernel / np.sum(kernel)

    output = np.zeros_like(
        image,
        dtype=float
    )

    rows, cols = image.shape

    for i in range(rows):

        for j in range(cols):

            region = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.sum(
                region * kernel
            )

    return np.uint8(
        np.clip(output, 0, 255)
    )

def gaussian_filter(
    image,
    filter_size,
    sigma,
    padding_type
):

    pad = filter_size // 2

    padded = apply_padding(
        image,
        pad,
        padding_type
    )

    kernel = np.zeros(
        (filter_size, filter_size),
        dtype=float
    )

    for x in range(-pad, pad + 1):

        for y in range(-pad, pad + 1):

            kernel[
                x + pad,
                y + pad
            ] = (
                1 / (2 * np.pi * sigma ** 2)
            ) * np.exp(
                -(x ** 2 + y ** 2) /
                (2 * sigma ** 2)
            )

    kernel = kernel / np.sum(kernel)

    output = np.zeros_like(
        image,
        dtype=float
    )

    rows, cols = image.shape

    for i in range(rows):

        for j in range(cols):

            region = padded[
                i:i + filter_size,
                j:j + filter_size
            ]

            output[i, j] = np.sum(
                region * kernel
            )

    return np.uint8(
        np.clip(output, 0, 255)
    )


print("=" * 60)
print("             IMAGE SMOOTHING PROGRAM")
print("=" * 60)

image_path = input(
    "\nEnter input image path: "
)

image = cv2.imread(
    image_path,
    cv2.IMREAD_GRAYSCALE
)

if image is None:

    print("\nERROR: Image not found!")
    print("Please check the image path.")

    exit()
print("\nImage loaded successfully!")
print("Image size:", image.shape)


while True:

    filter_size = int(
        input(
            "\nEnter filter size "
            "(3, 5, 7, ...): "
        )
    )

    if filter_size > 0 and filter_size % 2 == 1:

        break

    print(
        "Filter size must be a positive odd number."
    )

print("\nSelect Padding Method")
print("----------------------")
print("1. Zero Padding")
print("2. Replication Padding")
print("3. Reflection Padding")

padding_type = int(
    input(
        "Enter your choice (1-3): "
    )
)

while True:

    sigma = float(
        input(
            "\nEnter Gaussian sigma "
            "(e.g. 1.0): "
        )
    )

    if sigma > 0:

        break

    print("Sigma must be greater than zero.")
    
print("\n" + "=" * 60)
print("INPUT PARAMETERS")
print("=" * 60)

print("Filter Size       :", filter_size, "x", filter_size)

if padding_type == 1:
    print("Padding Method    : Zero Padding")

elif padding_type == 2:
    print("Padding Method    : Replication Padding")

elif padding_type == 3:
    print("Padding Method    : Reflection Padding")

print("Gaussian Sigma    :", sigma)

print("\nApplying Averaging Filter...")

average_result = averaging_filter(
    image,
    filter_size,
    padding_type
)

print("Averaging Filter completed.")

print("\nApplying Weighted Filter...")

weighted_result = weighted_filter(
    image,
    filter_size,
    padding_type
)

print("Weighted Filter completed.")

print("\nApplying Gaussian Filter...")

gaussian_result = gaussian_filter(
    image,
    filter_size,
    sigma,
    padding_type
)

print("Gaussian Filter completed.")

cv2.imwrite(
    "output_averaging.jpg",
    average_result
)

cv2.imwrite(
    "output_weighted.jpg",
    weighted_result
)

cv2.imwrite(
    "output_gaussian.jpg",
    gaussian_result
)

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)

plt.imshow(
    image,
    cmap='gray'
)

plt.title("Original Image")

plt.axis("off")

plt.subplot(2, 2, 2)

plt.imshow(
    average_result,
    cmap='gray'
)

plt.title(
    "Averaging Filter"
)

plt.axis("off")
plt.subplot(2, 2, 3)

plt.imshow(
    weighted_result,
    cmap='gray'
)

plt.title(
    "Weighted Filter"
)

plt.axis("off")

plt.subplot(2, 2, 4)

plt.imshow(
    gaussian_result,
    cmap='gray'
)

plt.title(
    "Gaussian Filter"
)

plt.axis("off")


plt.tight_layout()

plt.show()


print("\n" + "=" * 60)
print("              PROGRAM COMPLETED")
print("=" * 60)

print("\nOutput files:")
print("1. output_averaging.jpg")
print("2. output_weighted.jpg")
print("3. output_gaussian.jpg")

print("\nAll three smoothing filters were applied successfully.")